In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA01 - Travel, Lodging, and Entertainment
   
   DATA SOURCES:
   1. Master AU List: hive_metastore.ra_adido_2025.fy25_list_of_units
   2. Cost Center Mapping: vw_cost_center_mapping_bootstrap
   3. Coupa Data: 7 filtered files (ritm16070584_...)
   4. Finance Data: 6 files (f25_finance_data_...)
   
   BUSINESS LOGIC & ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + CANADA flag) will appear.
   2. SOURCE MERGE: Combines all 7 Coupa files and 6 Finance files into a single 
      unified transaction pipeline.
   3. STRING PARSING: Extracts the Cost Center and Category Code from Coupa's 
      hyphenated 'Account' string using split().
   4. TARGET CATEGORY CODES: Filters strictly for the ~30 approved target codes 
      provided in the master list.
   5. THE "793" EXCEPTION RULE: If a Category Code is 793, it is strictly locked to 
      Assessable Unit '101016'. If it maps to any other AU, it is intentionally dropped.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

CC_Mapping AS (
    -- 2. Pull the mapped Cost Centers from our finalized bootstrap view
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Combined_Coupa_Raw AS (
    -- 3a. Stack all 7 Coupa target files together
    SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    -- 3b. Extract the CC (1st block) and Category Code (3rd block) from Coupa's hyphenated string
    SELECT 
        Account AS Raw_String,
        CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0') ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END AS Cleaned_CC,
        TRY_CAST(TRIM(SPLIT(Account, '-')[2]) AS INT) AS CatCode_Int
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

Combined_Finance_Raw AS (
    -- 4a. Stack all 6 Finance target files together
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Finance_Parsed AS (
    -- 4b. Standardize Finance columns to match Coupa format
    SELECT 
        CostCenter AS Raw_String,
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRY_CAST(TRIM(CatCode) AS INT) AS CatCode_Int
    FROM Combined_Finance_Raw
),

All_Transactions AS (
    -- 5. Merge the parsed Coupa and Finance streams into one master transaction list
    SELECT Raw_String, Cleaned_CC, CatCode_Int FROM Coupa_Parsed
    UNION ALL
    SELECT Raw_String, Cleaned_CC, CatCode_Int FROM Finance_Parsed
),

Mapped_Transactions AS (
    -- 6. Bridge all transactions to AUs
    SELECT 
        t.Raw_String,
        t.Cleaned_CC,
        t.CatCode_Int,
        m.AU_ID
    FROM All_Transactions t
    INNER JOIN CC_Mapping m ON t.Cleaned_CC = m.Mapped_CC
),

Flagged_Engagements AS (
    -- 7. Apply the ~30 Category Code filters and the strict 793 exception rule
    SELECT 
        t.Raw_String,
        t.Cleaned_CC,
        t.AU_ID
    FROM Mapped_Transactions t
    WHERE 
        -- RULE 1: Must be in the approved master category list
        t.CatCode_Int IN (9, 12, 66, 67, 95, 134, 168, 192, 203, 204, 208, 209, 242, 269, 270, 484, 487, 636, 637, 638, 639, 676, 782, 783, 784, 792, 793, 890, 892)
        
        -- RULE 2: If the code is 793, it MUST be mapped to AU 101016 to be counted
        AND NOT (t.CatCode_Int = 793 AND t.AU_ID != '101016')
),

Engagements_By_AU AS (
    -- 8. Count the total valid flagged transactions per AU
    SELECT 
        AU_ID,
        COUNT(Raw_String) AS Match_Count
    FROM Flagged_Engagements
    GROUP BY AU_ID
)

-- 9. Final Output: Strictly structured per Master Anchor using count of valid findings
SELECT 
    mast.BusinessID, 
    mast.AU_Name, 
    mast.Business_Segment,
    'EBA01' AS QuestionID,               
    COALESCE(e.Match_Count, 0) AS Response,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN Engagements_By_AU e 
    ON mast.BusinessID = e.AU_ID
ORDER BY mast.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA01 - AU Level Calculation Review
   PURPOSE: One row per AU showing normalized Cost Centers and how the final count
   response was calculated.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Combined_Coupa_Raw AS (
    SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT Account FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Parsed AS (
    SELECT 
        CASE WHEN TRIM(SPLIT(Account, '-')[0]) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(SPLIT(Account, '-')[0]), 4), 4, '0') ELSE UPPER(TRIM(SPLIT(Account, '-')[0])) END AS Cleaned_CC,
        TRY_CAST(TRIM(SPLIT(Account, '-')[2]) AS INT) AS CatCode_Int,
        'Coupa' AS Source_System
    FROM Combined_Coupa_Raw
    WHERE Account LIKE '%-%-%'
),

Combined_Finance_Raw AS (
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Finance_Parsed AS (
    SELECT 
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRY_CAST(TRIM(CatCode) AS INT) AS CatCode_Int,
        'Finance' AS Source_System
    FROM Combined_Finance_Raw
),

All_Transactions AS (
    SELECT Cleaned_CC, CatCode_Int, Source_System FROM Coupa_Parsed
    UNION ALL
    SELECT Cleaned_CC, CatCode_Int, Source_System FROM Finance_Parsed
),

Mapped_Transactions AS (
    SELECT 
        m.AU_ID,
        t.Cleaned_CC,
        t.CatCode_Int,
        t.Source_System
    FROM All_Transactions t
    INNER JOIN CC_Mapping m
        ON t.Cleaned_CC = m.Mapped_CC
),

Target_CatCode_Transactions AS (
    SELECT *
    FROM Mapped_Transactions
    WHERE CatCode_Int IN (9, 12, 66, 67, 95, 134, 168, 192, 203, 204, 208, 209, 242, 269, 270, 484, 487, 636, 637, 638, 639, 676, 782, 783, 784, 792, 793, 890, 892)
),

Counted_Transactions AS (
    SELECT *
    FROM Target_CatCode_Transactions
    WHERE NOT (CatCode_Int = 793 AND AU_ID != '101016')
),

Mapped_CC_List AS (
    SELECT 
        AU_ID,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Cleaned_CC))) AS Normalized_CC_List,
        COUNT(*) AS Mapped_Row_Count
    FROM Mapped_Transactions
    GROUP BY AU_ID
),

Target_CatCode_By_AU AS (
    SELECT 
        AU_ID,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Cleaned_CC))) AS Target_CatCode_CC_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CAST(CatCode_Int AS STRING)))) AS Target_CatCode_List,
        COUNT(*) AS Target_CatCode_Row_Count
    FROM Target_CatCode_Transactions
    GROUP BY AU_ID
),

Counted_By_AU AS (
    SELECT 
        AU_ID,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Cleaned_CC))) AS Counted_CC_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CAST(CatCode_Int AS STRING)))) AS Counted_CatCode_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Source_System))) AS Source_System_List,
        COUNT(*) AS Counted_Row_Count
    FROM Counted_Transactions
    GROUP BY AU_ID
)

-- Output columns:
-- BusinessID: Master AU ID from the ABAC AU anchor list.
-- AU_Name: Master AU name tied to BusinessID.
-- Business_Segment: Business segment carried from the master AU list.
-- QuestionID: Questionnaire code for this debug output.
-- Response: Final count returned for EBA01 for this AU.
-- Normalized_CC_List: Normalized cost centers mapped to this AU before finance filtering.
-- Target_CatCode_CC_List: Cost centers that still have rows after the target category-code filter.
-- Counted_CC_List: Cost centers that contributed to the final counted rows.
-- Counted_CatCode_List: Category codes present on the final counted rows.
-- Source_System_List: Source-system values present on the final counted rows.
-- Mapped_Row_Count: Total mapped finance rows for this AU before the target-category count is finalized.
-- Target_CatCode_Row_Count: Rows that passed the target category-code step before the 793 exception rule.
-- Counted_Row_Count: Rows remaining after all EBA01 filters.
-- Final_Count: Duplicate of the final counted-row total used for troubleshooting.
-- Calculation_Formula: Plain-English explanation of how Response was produced.
-- In_ABAC_AU_List: Confirms the AU came from the standard ABAC AU list.
SELECT 
    mast.BusinessID,
    mast.AU_Name,
    mast.Business_Segment,
    'EBA01' AS QuestionID,
    COALESCE(c.Counted_Row_Count, 0) AS Response,
    COALESCE(m.Normalized_CC_List, 'n/a') AS Normalized_CC_List,
    COALESCE(t.Target_CatCode_CC_List, 'n/a') AS Target_CatCode_CC_List,
    COALESCE(c.Counted_CC_List, 'n/a') AS Counted_CC_List,
    COALESCE(c.Counted_CatCode_List, 'n/a') AS Counted_CatCode_List,
    COALESCE(c.Source_System_List, 'n/a') AS Source_System_List,
    COALESCE(m.Mapped_Row_Count, 0) AS Mapped_Row_Count,
    COALESCE(t.Target_CatCode_Row_Count, 0) AS Target_CatCode_Row_Count,
    COALESCE(c.Counted_Row_Count, 0) AS Counted_Row_Count,
    COALESCE(c.Counted_Row_Count, 0) AS Final_Count,
    CONCAT('Response = ', CAST(COALESCE(c.Counted_Row_Count, 0) AS STRING), ' counted rows after target category filter and 793 exception rule. Target-category rows=', CAST(COALESCE(t.Target_CatCode_Row_Count, 0) AS STRING), ', mapped rows=', CAST(COALESCE(m.Mapped_Row_Count, 0) AS STRING), '.') AS Calculation_Formula,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN Mapped_CC_List m
    ON mast.BusinessID = m.AU_ID
LEFT JOIN Target_CatCode_By_AU t
    ON mast.BusinessID = t.AU_ID
LEFT JOIN Counted_By_AU c
    ON mast.BusinessID = c.AU_ID
ORDER BY mast.BusinessID;


In [ ]:
%sql
/* ===================================================================================
   ROW-LEVEL DEBUG TABLE: EBA01 - AU 101011 Coupa Records
   PURPOSE: One row per Coupa source record that maps to AU 101011.
   Pulls ALL columns from the Coupa source tables alongside parsed/derived
   columns and filter evaluations.
=================================================================================== */

WITH CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Combined_Coupa_Raw AS (
    SELECT *, 'Coupa_JanFeb2025' AS Source_File FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT *, 'Coupa_MarApr2025' FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT *, 'Coupa_May2025' FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT *, 'Coupa_Jun2025' FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT *, 'Coupa_JulAug2025' FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT *, 'Coupa_SepOct2025' FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT *, 'Coupa_NovDec2024' FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
),

Coupa_Full AS (
    SELECT
        c.*,
        m.AU_ID,
        CASE WHEN TRIM(SPLIT(c.Account, '-')[0]) RLIKE '^[0-9]+$'
             THEN LPAD(RIGHT(TRIM(SPLIT(c.Account, '-')[0]), 4), 4, '0')
             ELSE UPPER(TRIM(SPLIT(c.Account, '-')[0]))
        END AS Cleaned_CC,
        TRIM(SPLIT(c.Account, '-')[2]) AS CatCode_Raw,
        TRY_CAST(TRIM(SPLIT(c.Account, '-')[2]) AS INT) AS CatCode_Int
    FROM Combined_Coupa_Raw c
    INNER JOIN CC_Mapping m
        ON (CASE WHEN TRIM(SPLIT(c.Account, '-')[0]) RLIKE '^[0-9]+$'
                 THEN LPAD(RIGHT(TRIM(SPLIT(c.Account, '-')[0]), 4), 4, '0')
                 ELSE UPPER(TRIM(SPLIT(c.Account, '-')[0]))
            END) = m.Mapped_CC
    WHERE c.Account LIKE '%-%-%'
      AND m.AU_ID = '101011'
)

-- Output columns:
-- Row_Num: Sequential row number for reference.
-- * (all columns from Coupa_Full): every raw source column + Source_File + AU_ID + Cleaned_CC + CatCode_Raw + CatCode_Int.
-- Is_Target_CatCode: Whether this CatCode is in the 29-code target list.
-- Exception_793_Pass: Whether this row passes the 793 exception rule.
-- Is_Counted: Whether this row is included in the final EBA01 count.
-- Exclusion_Reason: Why excluded, or 'Counted' if included.
SELECT
    ROW_NUMBER() OVER (ORDER BY Source_File, Account) AS Row_Num,
    *,
    CASE WHEN CatCode_Int IN (9,12,66,67,95,134,168,192,203,204,208,209,242,269,270,484,487,636,637,638,639,676,782,783,784,792,793,890,892) THEN 'Yes' ELSE 'No' END AS Is_Target_CatCode,
    CASE WHEN CatCode_Int = 793 AND AU_ID != '101016' THEN 'No' ELSE 'Yes' END AS Exception_793_Pass,
    CASE WHEN CatCode_Int IN (9,12,66,67,95,134,168,192,203,204,208,209,242,269,270,484,487,636,637,638,639,676,782,783,784,792,793,890,892)
          AND NOT (CatCode_Int = 793 AND AU_ID != '101016')
         THEN 'Yes' ELSE 'No'
    END AS Is_Counted,
    CASE
        WHEN CatCode_Int IS NULL THEN 'CatCode could not be parsed to integer'
        WHEN CatCode_Int NOT IN (9,12,66,67,95,134,168,192,203,204,208,209,242,269,270,484,487,636,637,638,639,676,782,783,784,792,793,890,892) THEN CONCAT('CatCode ', CAST(CatCode_Int AS STRING), ' not in target list')
        WHEN CatCode_Int = 793 AND AU_ID != '101016' THEN 'CatCode 793 excluded for non-101016 AU'
        ELSE 'Counted'
    END AS Exclusion_Reason
FROM Coupa_Full
ORDER BY Source_File, Account

In [ ]:
%sql
/* ===================================================================================
   ROW-LEVEL DEBUG TABLE: EBA01 - AU 101011 Finance Records
   PURPOSE: One row per Finance source record that maps to AU 101011.
   Pulls ALL columns from the Finance source tables alongside parsed/derived
   columns and filter evaluations.
=================================================================================== */

WITH CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Combined_Finance_Raw AS (
    SELECT *, 'Finance_File_1' AS Source_File FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT *, 'Finance_File_2' FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT *, 'Finance_File_3' FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT *, 'Finance_File_4' FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT *, 'Finance_File_5' FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT *, 'Finance_File_6' FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Finance_Full AS (
    SELECT
        f.*,
        m.AU_ID,
        CASE WHEN TRIM(f.CostCenter) RLIKE '^[0-9]+$'
             THEN LPAD(RIGHT(TRIM(f.CostCenter), 4), 4, '0')
             ELSE UPPER(TRIM(f.CostCenter))
        END AS Cleaned_CC,
        TRIM(f.CatCode) AS CatCode_Raw,
        TRY_CAST(TRIM(f.CatCode) AS INT) AS CatCode_Int
    FROM Combined_Finance_Raw f
    INNER JOIN CC_Mapping m
        ON (CASE WHEN TRIM(f.CostCenter) RLIKE '^[0-9]+$'
                 THEN LPAD(RIGHT(TRIM(f.CostCenter), 4), 4, '0')
                 ELSE UPPER(TRIM(f.CostCenter))
            END) = m.Mapped_CC
    WHERE m.AU_ID = '101011'
)

-- Output columns:
-- Row_Num: Sequential row number for reference.
-- * (all columns from Finance_Full): every raw source column + Source_File + AU_ID + Cleaned_CC + CatCode_Raw + CatCode_Int.
-- Is_Target_CatCode: Whether this CatCode is in the 29-code target list.
-- Exception_793_Pass: Whether this row passes the 793 exception rule.
-- Is_Counted: Whether this row is included in the final EBA01 count.
-- Exclusion_Reason: Why excluded, or 'Counted' if included.
SELECT
    ROW_NUMBER() OVER (ORDER BY Source_File, CostCenter, CatCode) AS Row_Num,
    *,
    CASE WHEN CatCode_Int IN (9,12,66,67,95,134,168,192,203,204,208,209,242,269,270,484,487,636,637,638,639,676,782,783,784,792,793,890,892) THEN 'Yes' ELSE 'No' END AS Is_Target_CatCode,
    CASE WHEN CatCode_Int = 793 AND AU_ID != '101016' THEN 'No' ELSE 'Yes' END AS Exception_793_Pass,
    CASE WHEN CatCode_Int IN (9,12,66,67,95,134,168,192,203,204,208,209,242,269,270,484,487,636,637,638,639,676,782,783,784,792,793,890,892)
          AND NOT (CatCode_Int = 793 AND AU_ID != '101016')
         THEN 'Yes' ELSE 'No'
    END AS Is_Counted,
    CASE
        WHEN CatCode_Int IS NULL THEN 'CatCode could not be parsed to integer'
        WHEN CatCode_Int NOT IN (9,12,66,67,95,134,168,192,203,204,208,209,242,269,270,484,487,636,637,638,639,676,782,783,784,792,793,890,892) THEN CONCAT('CatCode ', CAST(CatCode_Int AS STRING), ' not in target list')
        WHEN CatCode_Int = 793 AND AU_ID != '101016' THEN 'CatCode 793 excluded for non-101016 AU'
        ELSE 'Counted'
    END AS Exclusion_Reason
FROM Finance_Full
ORDER BY Source_File, CostCenter, CatCode

In [ ]:
# ===================================================================================
# EXPORT: EBA01 - AU 101011 Coupa Row-Level Debug -> CSV (PySpark robust export)
# Output path: dbfs:/tmp/eba01_au101011_coupa_debug.csv
# ===================================================================================
from pyspark.sql.functions import (
    col, lit, trim, when, lpad, upper, split, substring,
    row_number, concat
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

TARGET_CATCODES = [9,12,66,67,95,134,168,192,203,204,208,209,242,269,270,484,487,636,637,638,639,676,782,783,784,792,793,890,892]

coupa_tables = [
    ("hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered", "Coupa_JanFeb2025"),
    ("hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered", "Coupa_MarApr2025"),
    ("hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered", "Coupa_May2025"),
    ("hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered", "Coupa_Jun2025"),
    ("hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered", "Coupa_JulAug2025"),
    ("hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered", "Coupa_SepOct2025"),
    ("hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered", "Coupa_NovDec2024")
]

def normalized_cc_from_col(c):
    c_trim = trim(c)
    return when(
        c_trim.rlike(r"^[0-9]+$"),
        lpad(substring(c_trim, -4, 4), 4, "0")
    ).otherwise(upper(c_trim))

def safe_rm(path):
    try:
        dbutils.fs.rm(path, recurse=True)
    except Exception:
        pass

def export_single_csv(df, output_dir, final_path):
    safe_rm(output_dir)
    safe_rm(final_path)
    df.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_dir)
    part_file = [f.path for f in dbutils.fs.ls(output_dir) if f.name.startswith("part-")][0]
    dbutils.fs.cp(part_file, final_path)
    return final_path

# Build cost center mapping explicitly so export does not depend on notebook temp view state
df_cc_mapping = (
    spark.table("vw_cost_center_mapping_bootstrap")
    .select(
        trim(col("AU_ID").cast("string")).alias("AU_ID"),
        normalized_cc_from_col(col("cost_center_id")).alias("Mapped_CC")
    )
    .where(col("AU_ID").isNotNull() & col("cost_center_id").isNotNull())
    .distinct()
)

# Robustly union all source tables, allowing for month-to-month schema drift
df_coupa_raw = None
for table_name, source_file in coupa_tables:
    df_temp = spark.table(table_name).withColumn("Source_File", lit(source_file))
    df_coupa_raw = df_temp if df_coupa_raw is None else df_coupa_raw.unionByName(df_temp, allowMissingColumns=True)

account_parts = split(col("Account"), "-")

df_coupa = (
    df_coupa_raw
    .where(col("Account").like("%-%-%"))
    .withColumn("Cleaned_CC", normalized_cc_from_col(account_parts.getItem(0)))
    .withColumn("CatCode_Raw", trim(account_parts.getItem(2)))
    .withColumn("CatCode_Int", trim(account_parts.getItem(2)).cast(IntegerType()))
    .join(df_cc_mapping, col("Cleaned_CC") == col("Mapped_CC"), "inner")
    .where(col("AU_ID") == "101011")
    .withColumn(
        "Is_Target_CatCode",
        when(col("CatCode_Int").isin(TARGET_CATCODES), "Yes").otherwise("No")
    )
    .withColumn(
        "Exception_793_Pass",
        when((col("CatCode_Int") == 793) & (col("AU_ID") != "101016"), "No").otherwise("Yes")
    )
    .withColumn(
        "Is_Counted",
        when(
            col("CatCode_Int").isin(TARGET_CATCODES) &
            ~((col("CatCode_Int") == 793) & (col("AU_ID") != "101016")),
            "Yes"
        ).otherwise("No")
    )
    .withColumn(
        "Exclusion_Reason",
        when(col("CatCode_Int").isNull(), lit("CatCode could not be parsed to integer"))
        .when(~col("CatCode_Int").isin(TARGET_CATCODES), concat(lit("CatCode "), col("CatCode_Int").cast("string"), lit(" not in target list")))
        .when((col("CatCode_Int") == 793) & (col("AU_ID") != "101016"), lit("CatCode 793 excluded for non-101016 AU"))
        .otherwise(lit("Counted"))
    )
)

# Add row number at the end so ordering is deterministic in the export
w = Window.orderBy(col("Source_File"), col("Account"))
df_coupa = df_coupa.withColumn("Row_Num", row_number().over(w))

# Put Row_Num first for easier review
ordered_cols = ["Row_Num"] + [c for c in df_coupa.columns if c != "Row_Num"]
df_coupa = df_coupa.select(*ordered_cols)

output_dir = "dbfs:/tmp/eba01_au101011_coupa_debug"
final_path = "dbfs:/tmp/eba01_au101011_coupa_debug.csv"

export_single_csv(df_coupa, output_dir, final_path)

print(f"Exported: {final_path}")
print(f"Total rows: {df_coupa.count()}")


In [ ]:
# ===================================================================================
# EXPORT: EBA01 - AU 101011 Finance Row-Level Debug -> CSV (PySpark robust export)
# Output path: dbfs:/tmp/eba01_au101011_finance_debug.csv
# ===================================================================================
from pyspark.sql.functions import (
    col, lit, trim, when, lpad, upper, substring,
    row_number, concat
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType

TARGET_CATCODES = [9,12,66,67,95,134,168,192,203,204,208,209,242,269,270,484,487,636,637,638,639,676,782,783,784,792,793,890,892]

finance_tables = [
    ("hive_metastore.ra_adido_2025.f25_finance_data_1", "Finance_File_1"),
    ("hive_metastore.ra_adido_2025.f25_finance_data_2", "Finance_File_2"),
    ("hive_metastore.ra_adido_2025.f25_finance_data_3", "Finance_File_3"),
    ("hive_metastore.ra_adido_2025.f25_finance_data_4", "Finance_File_4"),
    ("hive_metastore.ra_adido_2025.f25_finance_data_5", "Finance_File_5"),
    ("hive_metastore.ra_adido_2025.f25_finance_data_6", "Finance_File_6")
]

def normalized_cc_from_col(c):
    c_trim = trim(c)
    return when(
        c_trim.rlike(r"^[0-9]+$"),
        lpad(substring(c_trim, -4, 4), 4, "0")
    ).otherwise(upper(c_trim))

def safe_rm(path):
    try:
        dbutils.fs.rm(path, recurse=True)
    except Exception:
        pass

def export_single_csv(df, output_dir, final_path):
    safe_rm(output_dir)
    safe_rm(final_path)
    df.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_dir)
    part_file = [f.path for f in dbutils.fs.ls(output_dir) if f.name.startswith("part-")][0]
    dbutils.fs.cp(part_file, final_path)
    return final_path

# Build cost center mapping explicitly so export does not depend on notebook temp view state
df_cc_mapping = (
    spark.table("vw_cost_center_mapping_bootstrap")
    .select(
        trim(col("AU_ID").cast("string")).alias("AU_ID"),
        normalized_cc_from_col(col("cost_center_id")).alias("Mapped_CC")
    )
    .where(col("AU_ID").isNotNull() & col("cost_center_id").isNotNull())
    .distinct()
)

# Robustly union all source tables allowing for missing columns
df_finance_raw = None
for table_name, source_file in finance_tables:
    df_temp = spark.table(table_name).withColumn("Source_File", lit(source_file))
    df_finance_raw = df_temp if df_finance_raw is None else df_finance_raw.unionByName(df_temp, allowMissingColumns=True)

df_finance = (
    df_finance_raw
    .withColumn("Cleaned_CC", normalized_cc_from_col(col("CostCenter")))
    .withColumn("CatCode_Raw", trim(col("CatCode")))
    .withColumn("CatCode_Int", trim(col("CatCode")).cast(IntegerType()))
    .join(df_cc_mapping, col("Cleaned_CC") == col("Mapped_CC"), "inner")
    .where(col("AU_ID") == "101011")
    .withColumn(
        "Is_Target_CatCode",
        when(col("CatCode_Int").isin(TARGET_CATCODES), "Yes").otherwise("No")
    )
    .withColumn(
        "Exception_793_Pass",
        when((col("CatCode_Int") == 793) & (col("AU_ID") != "101016"), "No").otherwise("Yes")
    )
    .withColumn(
        "Is_Counted",
        when(
            col("CatCode_Int").isin(TARGET_CATCODES) &
            ~((col("CatCode_Int") == 793) & (col("AU_ID") != "101016")),
            "Yes"
        ).otherwise("No")
    )
    .withColumn(
        "Exclusion_Reason",
        when(col("CatCode_Int").isNull(), lit("CatCode could not be parsed to integer"))
        .when(~col("CatCode_Int").isin(TARGET_CATCODES), concat(lit("CatCode "), col("CatCode_Int").cast("string"), lit(" not in target list")))
        .when((col("CatCode_Int") == 793) & (col("AU_ID") != "101016"), lit("CatCode 793 excluded for non-101016 AU"))
        .otherwise(lit("Counted"))
    )
)

# Add row number at the end so ordering is deterministic in the export
w = Window.orderBy(col("Source_File"), col("CostCenter"), col("CatCode"))
df_finance = df_finance.withColumn("Row_Num", row_number().over(w))

# Put Row_Num first for easier review
ordered_cols = ["Row_Num"] + [c for c in df_finance.columns if c != "Row_Num"]
df_finance = df_finance.select(*ordered_cols)

output_dir = "dbfs:/tmp/eba01_au101011_finance_debug"
final_path = "dbfs:/tmp/eba01_au101011_finance_debug.csv"

export_single_csv(df_finance, output_dir, final_path)

print(f"Exported: {final_path}")
print(f"Total rows: {df_finance.count()}")
